In [1]:
import numpy as np
from numba import njit,typed,types
from numba.experimental import jitclass

import smn
import stsp

In [2]:
@njit
def triangle(t,beta_T,beta_max):
    return beta_max*(t % beta_T)
@njit
def const(t,beta_T,beta_max):
    return beta_max
@njit
def step(t,beta_T,beta_max):
    return beta_max*(np.floor(t/beta_T)%2 == 0)

@njit
def arr_times_A_obeta(arr,cgu):
    #cgu - can go up
    res = -1.0*arr*cgu
    res[1:] += arr[:-1]*cgu[:-1]
    return res

@njit
def arr_times_A(arr,t,beta_f,beta_T,beta_max,cgu,A_fixed):
    beta = beta_f(t,beta_T,beta_max)
    return beta*arr_times_A_obeta(arr,cgu) + smn.array_times_sm(arr,A_fixed)

@njit
def evolve_RK(rho,t,dt,    beta_f,beta_T,beta_max,can_go_up,A_fixed):
    k1 = arr_times_A(rho            , t     , beta_f,beta_T,beta_max,can_go_up,A_fixed)
    k2 = arr_times_A(rho + k1*(dt/2), t+dt/2, beta_f,beta_T,beta_max,can_go_up,A_fixed)
    k3 = arr_times_A(rho + k2*(dt/2), t+dt/2, beta_f,beta_T,beta_max,can_go_up,A_fixed)
    k4 = arr_times_A(rho + k3*dt    , t+dt  , beta_f,beta_T,beta_max,can_go_up,A_fixed)

    return rho + (dt/6)*(k1+k2+k2+k3+k3+k4)
    
@njit
def RK_solve(init,t_init,t_final,dt,    beta_f,beta_T,beta_max,can_go_up,A_fixed):     
    t = t_init*1.0 #float
    rho = init*1.0 #array

    for i in range( int((t_final - t_init) / dt) ):
        rho = evolve_RK(rho,t,dt, beta_f,beta_T,beta_max,can_go_up,A_fixed)
        t+=dt
       
    return evolve_RK(rho,t,t_final-t, beta_f,beta_T,beta_max,can_go_up,A_fixed)

In [3]:
class A_triangle:
    def __init__(self,beta_max,beta_T,value_nbeta,can_go_up,N):
        S_ind = stsp.getS(N)
        rate_matrix_fixed = smn.sparse_matrix(*stsp.get_rate_matrix(np.concatenate([[0],value_nbeta])
                                                                    ,S_ind,N))

        self.A_fixed = rate_matrix_fixed - smn.sparse_matrix(np.arange(rate_matrix_fixed.shape,dtype=int),
                                                             np.arange(rate_matrix_fixed.shape,dtype=int),
                                                             rate_matrix_fixed.line_sum())
        
        self.beta_max = 1.0*beta_max
        self.beta_T = 1.0*beta_T
        self.can_go_up = can_go_up
        self.beta_f = triangle
    
    def arr_times_A(self,rho,t):
        return arr_times_A(rho,t,    self.beta_f,self.beta_T,self.beta_max,self.can_go_up,self.A_fixed)


    def evolve_RK(self,rho,t,dt):
        return evolve_RK(rho,t,dt,    self.beta_f,self.beta_T,self.beta_max,self.can_go_up,self.A_fixed)

    
    def solve(self,init,t_init,t_final,dt):
        return RK_solve(init,t_init,t_final,dt,    self.beta_f,self.beta_T,self.beta_max,self.can_go_up,self.A_fixed)


In [4]:
class case:
    def __init__(self,initial,value_nbeta,beta_max,function_params):

        self.value_nbeta = value_nbeta
        self.beta_max = beta_max
        self.beta_T = function_params
        #self.beta_function = lambda t: beta_function(t,beta_max,*function_params)

        gamma_s = value_nbeta[0]
        max_mean = beta_max/gamma_s

        Ns = int(max_mean + 10*np.sqrt(max_mean) + 1)
        N = np.array((4,2,Ns,Ns))
        self.N = N

        self.states = stsp.make_stsp(initial,N)
        self.pinitial = stsp.make_initial(initial,self.states)
        self.can_go_up = self.states[:,-1]!=(Ns-1)

        #S_ind = stsp.getS(N)
        self.A = A_triangle(beta_max,self.beta_T,value_nbeta,self.can_go_up,self.N)
        self.dt = .9/(np.abs(self.A.A_fixed.values).max()+beta_max)   

    
    def solver(self,t_init,T_finals,init=None):
        dt = self.dt
        if init is None:
            p = self.pinitial*1.0
        else:
            p = init*1.0
        if isinstance(1.0*T_finals, float) or isinstance(1.0*T_finals, np.float64) or isinstance(1.0*T_finals, np.float32):
            return self.A.solve(p,t_init,1.0*T_finals,dt)
        else:
            t = 1.0*t_init
            pt = []
            for T in T_finals:
                p = self.solver(t,T,p)
                pt.append(p)
                t=T
            return np.vstack(pt)

    #def run_guillespie(self,Ts):
    #    if isinstance(Ts,np.ndarray):
    #        return np.stack([self.run_guillespie(t)[1] for t in Ts])
    #    
    #    return guillespie(self.initial,Ts,self.value)


In [5]:
def create_cases(bog,allo_rate=10):
    init = np.array((0,  #A
                    0,  #B
                    0,  #P
                    bog*1.0 #S
                    ))
    
    allosteric_value = np.array((bog*1.0, #beta_s
                                1.,   #gamma_s
                                1,  #kAon
                                1,  #kAoff
                                10,  #kApon
                                1.,  #kApoff
                                1.,  #alpha
                                4.,  #alphap
                                10.,  #alpha_s
                                1.,  #alpha_sp
                                1.,  #nu
                                allo_rate*1.0, #nup
                                1., #kBon
                                1., #kBoff
                                1.  #gammaP
                                ))

    non_allost_value = np.array((bog*1.0, #beta_s
                                1.,   #gamma_s
                                11,  #kAon
                                2,  #kAoff
                                0,  #kApon
                                0,  #kApoff
                                0.,  #alpha
                                0,  #alphap
                                0.,  #alpha_s
                                0.,  #alpha_sp
                                (1.+allo_rate),  #nu
                                0., #nup
                                1., #kBon
                                1., #kBoff
                                1.  #gammaP
                                ))
    
    return case(init,allosteric_value[1:],bog,10.), case(init,non_allost_value[1:],bog,10.)


In [6]:
import params
bog = 4*(3.14159)
allo_rate = .7
c1,v = params.create_cases(bog,allo_rate)
c2,v = create_cases(bog,allo_rate)

In [7]:
c1.N,c1.states.dtype,c1.states

(array([ 4,  2, 49, 49]),
 dtype('int64'),
 array([[ 0,  0,  0,  0],
        [ 0,  0,  0,  1],
        [ 0,  0,  0,  2],
        ...,
        [ 3,  1, 48, 46],
        [ 3,  1, 48, 47],
        [ 3,  1, 48, 48]]))

In [8]:
c2.N,c2.states.dtype,c2.states

(array([ 4,  2, 49, 49]),
 dtype('int64'),
 array([[ 0,  0,  0,  0],
        [ 0,  0,  0,  1],
        [ 0,  0,  0,  2],
        ...,
        [ 3,  1, 48, 46],
        [ 3,  1, 48, 47],
        [ 3,  1, 48, 48]]))

In [9]:
c1.pinitial,c2.pinitial

(array([0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 7.0861678e-05,
        7.0861678e-05, 7.0861678e-05]),
 array([0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 7.0861678e-05,
        7.0861678e-05, 7.0861678e-05]))

In [10]:
v2_test = c2.solver(0.,np.arange(20))
v2_test

array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        7.08616780e-05, 7.08616780e-05, 7.08616780e-05],
       [1.36924880e-09, 1.36991804e-08, 7.14894258e-08, ...,
        1.32745986e-30, 3.41973581e-31, 8.00331737e-32],
       [2.38813584e-11, 3.13778327e-10, 2.16715351e-09, ...,
        9.13612577e-45, 3.28082853e-45, 1.15276603e-45],
       ...,
       [5.53613842e-35, 3.97823800e-33, 1.44829712e-31, ...,
        3.56191370e-72, 3.48571344e-72, 3.15175923e-72],
       [3.57977203e-38, 3.05326481e-36, 1.31648405e-34, ...,
        5.85385181e-72, 6.57573465e-72, 6.82753589e-72],
       [7.02429852e-41, 6.90281335e-39, 3.42401835e-37, ...,
        7.20656260e-72, 9.13739849e-72, 1.06944632e-71]])

In [11]:
v1_test = c1.solver(np.arange(20.))
v1_test

array([[0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
        7.08616780e-005, 7.08616780e-005, 7.08616780e-005],
       [3.81386365e-011, 4.87165395e-010, 3.24541140e-009, ...,
        9.00101018e-028, 2.84764874e-028, 7.75139784e-029],
       [7.46475379e-010, 7.67059852e-009, 4.22161166e-008, ...,
        8.00717273e-048, 2.15946568e-048, 5.15696477e-049],
       ...,
       [5.32463012e-008, 4.89009823e-007, 2.40523342e-006, ...,
        1.68257706e-103, 2.63612598e-104, 3.45188064e-105],
       [5.32537315e-008, 4.89078343e-007, 2.40557171e-006, ...,
        1.67828434e-103, 2.62936577e-104, 3.44297169e-105],
       [5.32577893e-008, 4.89115768e-007, 2.40575649e-006, ...,
        1.67594506e-103, 2.62568193e-104, 3.43811706e-105]])

In [12]:
np.max(v1_test-v2_test)

0.029010347503416835

In [13]:
arr = c2.solver(0.,np.arange(0,45,1.4))-c1.solver(np.arange(0,45,1.4))

arr.max()

0.15496855186242087

In [14]:
v1_test.sum()

19.999999999999716

In [15]:
v2_test.sum()

19.99999999999968

In [16]:
def step(t,beta_T,beta_max):
    return beta_max*(np.floor(t/beta_T)%2 == 0)

In [17]:
step(np.linspace(0,7,101),2,12)

array([12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12,
       12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12,
       12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12,
       12,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0])